In [2]:
import pandas as pd

In [10]:
bn= pd.read_csv('/Users/surisettivamsikrishna/Downloads/Vamsi Pc/CODES/Qoin/Cleaned data/bank_nifty_daily.csv')
zero_count = (bn['Volume']==0).sum()

In [11]:
print(zero_count)

1903


In [13]:
bn["Volume"].count()

4032

In [18]:
nifty = pd.read_csv('/Users/surisettivamsikrishna/Downloads/Vamsi Pc/CODES/Qoin/Cleaned data/nifty50_daily.csv')
zeros=(nifty["Volume"]==0).sum()
print(zeros)

763


In [19]:
"""
Fetch Bank Nifty (^NSEBANK) historical OHLCV from Yahoo Finance
and output it in the exact column format as bank_nifty_daily.csv:

    Date, Open, High, Low, Close, Volume, RSI_14

Run locally (not in a sandboxed environment):
    pip install yfinance pandas numpy
    python fetch_bank_nifty.py
"""

import pandas as pd
import numpy as np
import yfinance as yf

TICKER = "^NSEBANK"       # Bank Nifty index on Yahoo Finance
START = "2010-01-01"
END = "2026-06-03"        # one day past your last row (2026-06-02) so it's included
OUTPUT_FILE = "bank_nifty_yfinance.csv"


def compute_rsi(series: pd.Series, period: int = 14) -> pd.Series:
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi


def main():
    print(f"Downloading {TICKER} from {START} to {END} ...")
    df = yf.download(TICKER, start=START, end=END, interval="1d", auto_adjust=False, progress=False)

    if df.empty:
        raise SystemExit("No data returned. Yahoo Finance may not have volume history for this index that far back.")

    # yfinance sometimes returns MultiIndex columns when a single ticker is passed as a list-like; flatten if needed
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df = df.reset_index()  # brings Date out of the index

    df["RSI_14"] = compute_rsi(df["Close"], period=14)

    out = df[["Date", "Open", "High", "Low", "Close", "Volume", "RSI_14"]].copy()
    out["Date"] = pd.to_datetime(out["Date"]).dt.strftime("%Y-%m-%d")

    out.to_csv(OUTPUT_FILE, index=False)

    zero_vol = (out["Volume"] == 0).sum()
    print(f"Saved {len(out)} rows to {OUTPUT_FILE}")
    print(f"Rows with Volume == 0: {zero_vol} ({zero_vol / len(out):.1%})")
    print("Check this number for 2010-2012 specifically — Yahoo often has no real "
          "index volume for that period either, since index-level 'volume' isn't a "
          "traded quantity (only Bank Nifty futures/options volume is real).")


if __name__ == "__main__":
    main()

Saved 4044 rows to bank_nifty_yfinance.csv
Rows with Volume == 0: 1914 (47.3%)
Check this number for 2010-2012 specifically — Yahoo often has no real index volume for that period either, since index-level 'volume' isn't a traded quantity (only Bank Nifty futures/options volume is real).


Adding RSI to the files

In [21]:
import shutil
import pandas as pd

# ---- 1. Paths — fill these in with your actual local paths ----
MASTER_PATH   = "/Users/surisettivamsikrishna/Downloads/Vamsi Pc/CODES/Qoin/Model data/master_raw_aligned.csv"
NIFTY_PATH    = "/Users/surisettivamsikrishna/Downloads/Vamsi Pc/CODES/Qoin/Cleaned data/nifty50_daily.csv"        # image 1 (~5000-5200 range, 2010)
BANKNIFTY_PATH = "/Users/surisettivamsikrishna/Downloads/Vamsi Pc/CODES/Qoin/Cleaned data/bank_nifty_daily.csv"   # image 2 (~8900-9000 range, 2010) — same file as bank_nifty_daily.csv
ICICI_PATH    = "/Users/surisettivamsikrishna/Downloads/Vamsi Pc/CODES/Qoin/Cleaned data/icici_bank_daily.csv"        # image 3 (~150-160 range, real volume)

OUTPUT_PATH = MASTER_PATH  # overwrite the master file in place

# ---- 1b. Backup original master before overwriting ----
shutil.copy(MASTER_PATH, MASTER_PATH.replace(".csv", "_backup.csv"))

# ---- 2. Load master ----
master = pd.read_csv(MASTER_PATH)
master["Date"] = pd.to_datetime(master["Date"])

# ---- 3. Load each source, keep only Date + RSI_14, rename RSI_14 ----
def load_rsi(path: str, new_name: str) -> pd.DataFrame:
    df = pd.read_csv(path, usecols=["Date", "RSI_14"])
    df["Date"] = pd.to_datetime(df["Date"])
    return df.rename(columns={"RSI_14": new_name})

nifty_rsi     = load_rsi(NIFTY_PATH, "Nifty_RSI_14")
banknifty_rsi = load_rsi(BANKNIFTY_PATH, "BankNifty_RSI_14")
icici_rsi     = load_rsi(ICICI_PATH, "ICICI_RSI_14")

# ---- 4. Merge onto master (left join keeps every master row/date) ----
merged = (
    master
    .merge(nifty_rsi, on="Date", how="left")
    .merge(banknifty_rsi, on="Date", how="left")
    .merge(icici_rsi, on="Date", how="left")
)

# ---- 5. Sanity checks ----
print("Master rows:", len(master))
print("Merged rows:", len(merged))  # should match master rows exactly if join is clean

for col in ["Nifty_RSI_14", "BankNifty_RSI_14", "ICICI_RSI_14"]:
    missing = merged[col].isna().sum()
    print(f"{col}: {missing} missing after merge ({missing/len(merged):.1%})")

# ---- 6. Save ----
merged.to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH}")

Master rows: 4266
Merged rows: 4266
Nifty_RSI_14: 266 missing after merge (6.2%)
BankNifty_RSI_14: 250 missing after merge (5.9%)
ICICI_RSI_14: 246 missing after merge (5.8%)
Saved: /Users/surisettivamsikrishna/Downloads/Vamsi Pc/CODES/Qoin/Model data/master_raw_aligned.csv


## Dealing with FOrward Fills

"""
QOIN - Lookahead bias fix
Shifts period-start-dated macro/fundamental columns to their real-world
public release dates, using fixed institutional lag rules (since no
published-date column exists in the source data).

Columns fixed: CPI_Index, WPI_Index, Trade_Balance_Billion_USD,
               Real_GDP_YoY_Growth_%, Net Profit (Cr Cr), NII (Cr Cr)
Columns left untouched: Repo_Rate_%, CRR_% (announced same-day, effective
               same-day), FII_Net, DII_Net, market prices, RSI (already
               real-time / same-day data, no lag issue).
"""

In [22]:
import csv
from datetime import datetime, timedelta

IN_PATH = "/Users/surisettivamsikrishna/Downloads/Vamsi Pc/CODES/Qoin/Model data/master_raw_aligned.csv"
OUT_PATH = "master_aligned_fixed.csv"


LAGS = {
    "CPI_Index": 42,
    "WPI_Index": 44,
    "Trade_Balance_Billion_USD": 40,
    "Real_GDP_YoY_Growth_%": 60,
    "Net Profit (\u20b9 Cr)": 40,
    "NII (\u20b9 Cr)": 40,
}

def parse_date(s):
    return datetime.strptime(s, "%Y-%m-%d")

def main():
    with open(IN_PATH, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames
        rows = list(reader)

    trading_dates = [parse_date(r["Date"]) for r in rows]

    for col, lag_days in LAGS.items():
        # 1. detect change events in original series (date, value)
        change_events = []
        prev_val = None
        for r in rows:
            v = r[col]
            if v != prev_val:
                change_events.append((parse_date(r["Date"]), v))
                prev_val = v

        # 2. shift each change event's date forward by the lag
        shifted_events = [(d + timedelta(days=lag_days), v) for d, v in change_events]

        # 3. re-forward-fill onto the trading calendar using shifted dates
        #    before the first shifted event, use the original first value
        #    (best available assumption for the start of history)
        new_series = []
        event_idx = 0
        current_val = shifted_events[0][1]
        for td in trading_dates:
            while (event_idx < len(shifted_events) and
                   shifted_events[event_idx][0] <= td):
                current_val = shifted_events[event_idx][1]
                event_idx += 1
            new_series.append(current_val)

        for r, v in zip(rows, new_series):
            r[col] = v

    with open(OUT_PATH, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    print(f"Wrote {OUT_PATH} with {len(rows)} rows.")

if __name__ == "__main__":
    main()

Wrote master_aligned_fixed.csv with 4253 rows.


The master data which has been generated(master raw aligned csv) has more then two decimals for each fields so rounding off them to just two

In [2]:
 """
Convert each column to the RIGHT transform based on its type:
  - Prices (ICICI, Nifty, BankNifty, Gold, USD_INR)      -> % change
  - Rates/yields (Repo, CRR, Bond_10Y/5Y, US_10Y/5Y)       -> simple difference (bps-like)
  - Flow values that can be negative/near-zero (FII, DII)  -> simple difference
  - Levels that are already bounded/indices (RSI, CPI, WPI,
    Trade_Balance, GDP growth, Net Profit, NII)             -> keep raw level
      (CPI/WPI/GDP are usually monthly-frequency forward-filled;
       taking pct_change on daily-ffilled data creates fake 0-then-spike noise,
       so we leave them as levels here. If you have a proper MoM/YoY series,
       swap that in instead.)

No dropna() on the full frame - each column's NaNs (leading NaN from pct_change/diff,
or genuine missing data) are left in place. This preserves row count; handle
NaNs per-column at analysis time (e.g. pairwise-complete correlation) instead
of wiping whole rows.
"""
import pandas as pd

IN_PATH = "/Users/surisettivamsikrishna/Downloads/Vamsi Pc/CODES/Qoin/Model data/master_raw_aligned.csv"
OUT_PATH = "master_transformed.csv"

# Price-like columns -> % change
PCT_CHANGE_COLS = [
    "ICICI_Close", "Nifty_Close", "BankNifty_Close", "Gold_Close", "USD_INR",
]

# Rate/yield columns -> simple difference (NOT pct_change; a rate isn't a price)
DIFF_COLS = [
    "Bond_10Y", "Bond_5Y", "US_10Y", "US_5Y", "Repo_Rate_%", "CRR_%",
]

# Flow columns that can be negative/near-zero -> simple difference
FLOW_DIFF_COLS = [
    "FII_Net", "DII_Net",
]

# Already-level / bounded / low-frequency columns -> keep as-is
LEVEL_COLS = [
    "Nifty_RSI_14", "BankNifty_RSI_14", "ICICI_RSI_14",
    "CPI_Index", "WPI_Index", "Trade_Balance_Billion_USD",
    "Real_GDP_YoY_Growth_%", "Net Profit (₹ Cr)", "NII (₹ Cr)",
]

df = pd.read_csv(IN_PATH, parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# Round every numeric column to 2 decimals (Date is untouched since it's not numeric)
numeric_cols = df.select_dtypes(include="number").columns
df[numeric_cols] = df[numeric_cols].round(2)

out = pd.DataFrame({"Date": df["Date"]})

for col in df.columns:
    if col == "Date":
        continue

    if col in PCT_CHANGE_COLS:
        out[col] = df[col].pct_change() * 100          # % change

    elif col in DIFF_COLS:
        out[col] = df[col].diff()                       # simple difference

    elif col in FLOW_DIFF_COLS:
        out[col] = df[col].diff()                       # simple difference (avoids /0 blowups)

    elif col in LEVEL_COLS:
        out[col] = df[col]                               # raw level, untouched

    else:
        # Unrecognized column - default to raw level and flag it, rather than
        # silently guessing a transform that might be wrong.
        print(f"[warn] '{col}' not classified in any list - kept as raw level. "
              f"Check if it needs pct_change or diff instead.")
        out[col] = df[col]

# NOTE: no dropna() here on purpose.
# - pct_change()/diff() naturally produce NaN on row 0 for the transformed columns.
# - Any pre-existing missing data (e.g. monthly CPI ffilled with gaps) stays NaN too.
# Handle this at analysis time, e.g.:
#     out[["Gold_Close", "ICICI_Close"]].corr()   # pandas .corr() uses pairwise-complete by default
# rather than dropping entire rows here and losing unrelated columns' data.

# Round transformed values too (pct_change/diff often produce long float tails)
out_numeric_cols = out.select_dtypes(include="number").columns
out[out_numeric_cols] = out[out_numeric_cols].round(2)

out.to_csv(OUT_PATH, index=False)
print(f"Wrote {OUT_PATH} with {len(out)} rows (no rows dropped for NaNs).")
print(f"NaN count per column:\n{out.isna().sum()}")

Wrote master_transformed.csv with 4253 rows (no rows dropped for NaNs).
NaN count per column:
Date                           0
ICICI_Close                    1
Nifty_Close                    1
BankNifty_Close                1
Gold_Close                     1
USD_INR                        1
Bond_10Y                       1
Bond_5Y                        1
US_10Y                         1
US_5Y                          1
FII_Net                        1
DII_Net                        1
CPI_Index                      0
WPI_Index                      0
Trade_Balance_Billion_USD      0
Repo_Rate_%                    1
CRR_%                          1
Real_GDP_YoY_Growth_%          0
Net Profit (₹ Cr)              0
NII (₹ Cr)                     0
Nifty_RSI_14                 253
BankNifty_RSI_14             237
ICICI_RSI_14                 233
dtype: int64
